# Predicción de facturación diaria — comparación y validación de 5 modelos

## Objetivo
Predecir `facturacion` diaria y comparar cinco modelos de regresión:

1. Regresión Lineal
2. Random Forest
3. Gradient Boosting
4. HistGradientBoosting
5. XGBoost

El notebook utiliza **validación cruzada temporal (TimeSeriesSplit)** y un **test final separado cronológicamente**.

### Muy importante: evitar data leakage
No se utilizan variables que sean resultado de las ventas del propio día (`num_tickets`, `ticket_medio`, `ticket_mediano`) ni métricas de reservas que solo se conocen después de que termina el día.

La validación temporal es importante porque queremos simular la situación real: entrenar con pasado y predecir futuro.


In [2]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor
)

from sklearn.model_selection import TimeSeriesSplit, cross_validate
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    make_scorer
)

from xgboost import XGBRegressor

print("Librerías cargadas correctamente.")


Librerías cargadas correctamente.


## 1. Cargar el DataFrame del EDA



In [5]:
tabla_maestra = pd.read_parquet(
    "C:\\Users\\CandelaGB\\Desktop\\TFM anita\\TFM-Hosteleria-AI\\data\\gold\\tabla_maestra_diaria.parquet"
)
df = tabla_maestra.copy()

print("Dimensiones:", df.shape)
print("\nColumnas:")
print(df.columns)


Dimensiones: (242, 34)

Columnas:
Index(['fecha', 'num_tickets', 'facturacion', 'ticket_medio', 'ticket_mediano',
       'dia_semana', 'num_dia', 'mes', 'es_festivo', 'festivo_nombre',
       'tiene_evento', 'intensidad_evento', 'impacto_evento',
       'direccion_evento', 'categoria_evento', 'cat_evento',
       'n_reservas_cancelada', 'n_reservas_completada', 'n_reservas_no_show',
       'comensales_cancelada', 'comensales_completada', 'comensales_no_show',
       'temperature_max', 'temperature_min', 'temperature_mean',
       'precipitation_mm', 'precipitation_hours', 'wind_speed_max',
       'sunshine_duration_h', 'n_reservas_total', 'tasa_no_show',
       'tasa_cancelacion', 'es_fin_de_semana', 'es_lunes'],
      dtype='str')


In [6]:
df

,fecha,num_tickets,facturacion,ticket_medio,ticket_mediano,dia_semana,num_dia,mes,es_festivo,festivo_nombre,...,temperature_mean,precipitation_mm,precipitation_hours,wind_speed_max,sunshine_duration_h,n_reservas_total,tasa_no_show,tasa_cancelacion,es_fin_de_semana,es_lunes
0,2025-10-02,21,1339.33,63.777619,65.300,Thursday,3,10,0,sin_festivo,...,20.4,0.0,0.0,11.6,11.565747,17.0,0.00,5.88,0,0
1,2025-10-03,53,4498.50,84.877358,74.150,Friday,4,10,0,sin_festivo,...,19.8,0.0,0.0,11.5,11.521197,42.0,14.29,7.14,1,0
2,2025-10-04,40,4323.00,108.075000,97.375,Saturday,5,10,0,sin_festivo,...,21.2,0.0,0.0,17.5,11.304900,36.0,0.00,5.56,1,0
3,2025-10-05,16,1866.75,116.671875,128.500,Sunday,6,10,0,sin_festivo,...,20.0,0.0,0.0,17.3,11.371650,14.0,7.14,7.14,0,0
4,2025-10-07,27,1701.25,63.009259,65.000,Tuesday,1,10,0,sin_festivo,...,20.5,0.0,0.0,9.4,11.166731,20.0,5.00,0.00,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237,2026-07-04,33,3251.55,98.531818,89.000,Saturday,5,7,0,sin_festivo,...,30.5,0.0,0.0,21.9,14.491961,30.0,0.00,10.00,1,0
238,2026-07-05,12,1728.35,144.029167,116.375,Sunday,6,7,0,sin_festivo,...,31.0,0.0,0.0,13.9,14.302794,11.0,0.00,9.09,0,0
239,2026-07-07,21,2119.25,100.916667,63.200,Tuesday,1,7,0,sin_festivo,...,30.6,1.1,1.0,17.6,13.644478,16.0,0.00,0.00,0,0
240,2026-07-08,31,2183.20,70.425806,70.000,Wednesday,2,7,0,sin_festivo,...,29.1,0.0,0.0,13.8,14.686733,24.0,0.00,8.33,0,0


In [8]:
df["anio"] = df["fecha"].dt.year
df["semana_anio"] = df["fecha"].dt.isocalendar().week.astype(int)
df["trimestre"] = df["fecha"].dt.quarter
df["dia_anio"] = df["fecha"].dt.dayofyear

# Variables cíclicas: permiten representar la naturaleza circular de semana/año
df["sin_dia_anio"] = np.sin(2 * np.pi * df["dia_anio"] / 365.25)
df["cos_dia_anio"] = np.cos(2 * np.pi * df["dia_anio"] / 365.25)

display(df[[
    "fecha", "anio", "num_dia", "semana_anio",
    "trimestre", "dia_anio", "sin_dia_anio", "cos_dia_anio"
]].head())


,fecha,anio,num_dia,semana_anio,trimestre,dia_anio,sin_dia_anio,cos_dia_anio
0,2025-10-02,2025,3,40,4,275,-0.999833,0.018277
1,2025-10-03,2025,4,40,4,276,-0.999371,0.035473
2,2025-10-04,2025,5,40,4,277,-0.998613,0.052658
3,2025-10-05,2025,6,40,4,278,-0.997559,0.069828
4,2025-10-07,2025,1,41,4,280,-0.994567,0.104101


## 4. Selección de variables y control de leakage

### Variables excluidas
Estas variables no deben utilizarse si queremos predecir la facturación **antes de que termine el día**:

- `facturacion` → es el objetivo.
- `num_tickets` → solo se conoce al finalizar el día.
- `ticket_medio` → depende de las ventas del día.
- `ticket_mediano` → depende de las ventas del día.
- reservas completadas/canceladas/no-show → son resultados del día.
- `n_reservas_total` y tasas → se excluyen de forma conservadora hasta comprobar si representan información conocida antes del día.

### Variables potencialmente útiles
Sí utilizamos, cuando están disponibles:

- día de semana
- festivos
- eventos
- meteorología
- variables temporales
- otras variables externas conocidas antes del día

> Si `n_reservas_total` significa **reservas confirmadas que ya existían antes de comenzar el día**, podremos incorporarla en una segunda versión. Eso puede mejorar mucho la predicción.


In [9]:
target = "facturacion"

leakage_cols = [
    "facturacion",
    "num_tickets",
    "ticket_medio",
    "ticket_mediano",
    "n_reservas_cancelada",
    "n_reservas_completada",
    "n_reservas_no_show",
    "comensales_cancelada",
    "comensales_completada",
    "comensales_no_show",
    "n_reservas_total",
    "tasa_no_show",
    "tasa_cancelacion"
]

date_cols = ["fecha"]

feature_cols = [
    c for c in df.columns
    if c not in leakage_cols + date_cols
]

X = df[feature_cols].copy()
y = df[target].copy()

print("Número de variables:", len(feature_cols))
print("\nVariables utilizadas:")
for c in feature_cols:
    print(" -", c)


Número de variables: 26

Variables utilizadas:
 - dia_semana
 - num_dia
 - mes
 - es_festivo
 - festivo_nombre
 - tiene_evento
 - intensidad_evento
 - impacto_evento
 - direccion_evento
 - categoria_evento
 - cat_evento
 - temperature_max
 - temperature_min
 - temperature_mean
 - precipitation_mm
 - precipitation_hours
 - wind_speed_max
 - sunshine_duration_h
 - es_fin_de_semana
 - es_lunes
 - anio
 - semana_anio
 - trimestre
 - dia_anio
 - sin_dia_anio
 - cos_dia_anio


In [11]:
# 1. Rango de temperatura del día
df["rango_temperatura"] = (
    df["temperature_max"] - df["temperature_min"]
)

# 2. Día lluvioso
# Consideramos que ha llovido si la precipitación es mayor que 0 mm
df["es_dia_lluvioso"] = (
    df["precipitation_mm"] > 0
).astype(int)

# 3. Día muy caluroso
# Umbral inicial: temperatura máxima >= 30 ºC
df["es_dia_muy_caluroso"] = (
    df["temperature_max"] >= 30
).astype(int)

# 4. Día muy frío
# Umbral inicial: temperatura mínima <= 5 ºC
df["es_dia_muy_frio"] = (
    df["temperature_min"] <= 5
).astype(int)

In [12]:
# ==========================================
# VARIABLES TEMPORALES DE FACTURACIÓN
# ==========================================

# Facturación del día anterior
df["facturacion_1d"] = (
    df["facturacion"].shift(1)
)


# Media de facturación de los últimos 7 días
# (sin incluir el propio día)
df["facturacion_media_7d"] = (
    df["facturacion"]
    .shift(1)
    .rolling(window=7)
    .mean()
)


# Media de facturación de los últimos 30 días
# (sin incluir el propio día)
df["facturacion_media_30d"] = (
    df["facturacion"]
    .shift(1)
    .rolling(window=30)
    .mean()
)


# Suma de facturación de los últimos 7 días
df["facturacion_total_7d"] = (
    df["facturacion"]
    .shift(1)
    .rolling(window=7)
    .sum()
)


# Suma de facturación de los últimos 30 días
df["facturacion_total_30d"] = (
    df["facturacion"]
    .shift(1)
    .rolling(window=30)
    .sum()
)


# Desviación típica de facturación últimos 7 días
df["facturacion_std_7d"] = (
    df["facturacion"]
    .shift(1)
    .rolling(window=7)
    .std()
)


# Desviación típica de facturación últimos 30 días
df["facturacion_std_30d"] = (
    df["facturacion"]
    .shift(1)
    .rolling(window=30)
    .std()
)

In [13]:
df.columns

Index(['fecha', 'num_tickets', 'facturacion', 'ticket_medio', 'ticket_mediano',
       'dia_semana', 'num_dia', 'mes', 'es_festivo', 'festivo_nombre',
       'tiene_evento', 'intensidad_evento', 'impacto_evento',
       'direccion_evento', 'categoria_evento', 'cat_evento',
       'n_reservas_cancelada', 'n_reservas_completada', 'n_reservas_no_show',
       'comensales_cancelada', 'comensales_completada', 'comensales_no_show',
       'temperature_max', 'temperature_min', 'temperature_mean',
       'precipitation_mm', 'precipitation_hours', 'wind_speed_max',
       'sunshine_duration_h', 'n_reservas_total', 'tasa_no_show',
       'tasa_cancelacion', 'es_fin_de_semana', 'es_lunes', 'anio',
       'semana_anio', 'trimestre', 'dia_anio', 'sin_dia_anio', 'cos_dia_anio',
       'rango_temperatura', 'es_dia_lluvioso', 'es_dia_muy_caluroso',
       'es_dia_muy_frio', 'facturacion_1d', 'facturacion_media_7d',
       'facturacion_media_30d', 'facturacion_total_7d',
       'facturacion_total_30d',

In [24]:
df["festivo_nombre"].unique()

<ArrowStringArray>
[                     'sin_festivo',                 'Todos los Santos',
  'Día de la Constitución Española',            'Inmaculada Concepción',
              'Natividad del Señor',                        'Año Nuevo',
               'Epifanía del Señor',                     'Jueves Santo',
                    'Viernes Santo',               'Fiesta del Trabajo',
 'Fiesta de la Comunidad de Madrid']
Length: 11, dtype: str

## 5. Comprobar calidad de los datos

Antes de entrenar, revisamos nulos, infinitos, tipos y duplicados.


In [ ]:
print("Duplicados exactos:", df.duplicated().sum())

print("\nNulos:")
display(
    df[feature_cols + [target]]
    .isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("nulos")
    .query("nulos > 0")
)

print("\nValores infinitos:")
numeric_check = df[feature_cols + [target]].select_dtypes(include=np.number)
print(np.isinf(numeric_check).sum().sum())

print("\nVariable objetivo:")
display(df[target].describe())


## 6. Train/Test temporal

Reservamos el **20% final de las fechas como test final**.

El test final NO se utiliza para elegir el modelo ni para ajustar hiperparámetros.

Esto es importante: primero seleccionamos usando únicamente el período de entrenamiento mediante validación temporal y, al final, evaluamos una única vez sobre datos futuros nunca vistos.


In [ ]:
split = int(len(df) * 0.80)

X_train = X.iloc[:split].copy()
X_test = X.iloc[split:].copy()

y_train = y.iloc[:split].copy()
y_test = y.iloc[split:].copy()

print("TRAIN")
print(df["fecha"].iloc[0], "→", df["fecha"].iloc[split - 1])
print("Filas:", len(X_train))

print("\nTEST FINAL")
print(df["fecha"].iloc[split], "→", df["fecha"].iloc[-1])
print("Filas:", len(X_test))


## 7. Preprocesamiento

Creamos dos transformaciones:

- Para modelos lineales: imputación + estandarización.
- Para modelos de árboles: imputación sin necesidad de escalar.

Las transformaciones se encuentran dentro de los `Pipeline`, por lo que **en cada fold se ajustan únicamente con los datos de entrenamiento de ese fold**. Esto evita leakage durante la validación cruzada.


In [ ]:
categorical_cols = X_train.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

numeric_cols = X_train.select_dtypes(
    include=[np.number]
).columns.tolist()

numeric_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

numeric_tree = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocessor_linear = ColumnTransformer([
    ("num", numeric_scaled, numeric_cols),
    ("cat", categorical, categorical_cols)
])

preprocessor_tree = ColumnTransformer([
    ("num", numeric_tree, numeric_cols),
    ("cat", categorical, categorical_cols)
])

print("Numéricas:", numeric_cols)
print("\nCategóricas:", categorical_cols)


# 8. Los cinco modelos

Vamos a comparar:

### 1. Ridge
Modelo lineal regularizado. Es un buen **baseline**: si un modelo complejo no mejora a Ridge, probablemente no esté aportando demasiado.

### 2. Random Forest
Captura relaciones no lineales e interacciones entre variables.

### 3. Gradient Boosting
Construye árboles secuencialmente para corregir errores anteriores.

### 4. HistGradientBoosting
Versión eficiente de gradient boosting de scikit-learn.

### 5. XGBoost
Modelo de boosting muy potente para datos tabulares y uno de los candidatos principales para este problema.


In [ ]:
models = {
    "Ridge": Pipeline([
        ("preprocessor", preprocessor_linear),
        ("model", Ridge(alpha=10.0))
    ]),

    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", RandomForestRegressor(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=2,
            max_features=0.8,
            random_state=42,
            n_jobs=-1
        ))
    ]),

    "Gradient Boosting": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", GradientBoostingRegressor(
            n_estimators=300,
            learning_rate=0.03,
            max_depth=3,
            min_samples_leaf=3,
            loss="huber",
            random_state=42
        ))
    ]),

    "HistGradientBoosting": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", HistGradientBoostingRegressor(
            max_iter=300,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42
        ))
    ]),

    "XGBoost": Pipeline([
        ("preprocessor", preprocessor_tree),
        ("model", XGBRegressor(
            n_estimators=500,
            learning_rate=0.03,
            max_depth=4,
            min_child_weight=3,
            subsample=0.8,
            colsample_bytree=0.8,
            reg_alpha=0.1,
            reg_lambda=2.0,
            objective="reg:squarederror",
            eval_metric="mae",
            random_state=42,
            n_jobs=-1
        ))
    ])
}

print("Modelos preparados:")
print(list(models.keys()))


# 9. Validación cruzada temporal

Utilizamos `TimeSeriesSplit`.

No queremos folds aleatorios porque eso permitiría entrenar con información temporalmente posterior a la validación.

Cada fold se parece a:

```text
Fold 1: TRAIN →→ TEST
Fold 2: TRAIN →→→ TEST
Fold 3: TRAIN →→→→ TEST
Fold 4: TRAIN →→→→→ TEST
Fold 5: TRAIN →→→→→→ TEST
```

Además, dejamos un **test final completamente separado**.

Evaluaremos:

- MAE
- RMSE
- R²
- MAPE

**Nota:** MAPE puede ser problemático si la facturación contiene ceros. Por eso también calculamos sMAPE.


In [ ]:
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred))
    mask = denominator != 0
    return np.mean(
        2 * np.abs(y_pred[mask] - y_true[mask]) / denominator[mask]
    ) * 100

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2"
}

cv_results = []

for name, model in models.items():
    print(f"Validando {name}...")

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=tscv,
        scoring=scoring,
        n_jobs=1,
        return_train_score=True
    )

    cv_results.append({
        "modelo": name,
        "MAE_CV_mean": -scores["test_MAE"].mean(),
        "MAE_CV_std": scores["test_MAE"].std(),
        "RMSE_CV_mean": -scores["test_RMSE"].mean(),
        "RMSE_CV_std": scores["test_RMSE"].std(),
        "R2_CV_mean": scores["test_R2"].mean(),
        "R2_CV_std": scores["test_R2"].std(),
        "MAE_train_mean": -scores["train_MAE"].mean()
    })

cv_results_df = pd.DataFrame(cv_results).sort_values("MAE_CV_mean")

display(cv_results_df)


## 10. ¿Cuál es el mejor modelo en validación?

Como criterio principal usamos **MAE**, porque es fácil de interpretar:

> MAE = error medio absoluto en euros.

Cuanto menor sea, mejor.

También miraremos RMSE y R². Un modelo que tenga un MAE bajo pero un RMSE muchísimo mayor puede estar cometiendo algunos errores extremos.


In [ ]:
best_model_name = cv_results_df.iloc[0]["modelo"]

print("Mejor modelo según MAE medio de validación temporal:")
print(best_model_name)

print("\nResultados ordenados por MAE:")
display(cv_results_df)


# 11. Validación fold a fold

No basta con mirar la media. Comprobamos cómo se comporta cada modelo en cada período temporal.

Un modelo robusto debería mantener un rendimiento razonablemente estable.


In [ ]:
fold_rows = []

for name, model in models.items():
    fold_num = 1

    for train_idx, val_idx in tscv.split(X_train):
        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model.fit(X_tr, y_tr)
        pred = model.predict(X_val)

        fold_rows.append({
            "modelo": name,
            "fold": fold_num,
            "fecha_inicio_validacion": df["fecha"].iloc[val_idx[0]],
            "fecha_fin_validacion": df["fecha"].iloc[val_idx[-1]],
            "MAE": mean_absolute_error(y_val, pred),
            "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
            "R2": r2_score(y_val, pred),
            "sMAPE_%": smape(y_val.values, pred)
        })

        fold_num += 1

fold_results = pd.DataFrame(fold_rows)

display(
    fold_results.sort_values(["fold", "MAE"])
)


## 12. Estabilidad de los modelos

Calculamos la desviación estándar del MAE entre folds.

- **MAE medio bajo** → buen error medio.
- **MAE std bajo** → comportamiento más estable.
- Un modelo con un MAE ligeramente mejor pero una variabilidad enorme puede ser menos fiable.


In [ ]:
stability = (
    fold_results
    .groupby("modelo")
    .agg(
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        RMSE_mean=("RMSE", "mean"),
        R2_mean=("R2", "mean"),
        sMAPE_mean=("sMAPE_%", "mean")
    )
    .sort_values("MAE_mean")
)

display(stability)


# 13. Entrenamiento final y test futuro

Ahora entrenamos cada modelo con **todo el 80% inicial**.

El 20% final se mantiene intacto hasta este momento.

Esta será nuestra comparación final sobre datos que el modelo nunca ha visto.


In [ ]:
test_results = []
test_predictions = {}

for name, model in models.items():
    print(f"Entrenando y evaluando {name}...")

    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    test_predictions[name] = pred

    test_results.append({
        "modelo": name,
        "MAE_€": mean_absolute_error(y_test, pred),
        "RMSE_€": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
        "sMAPE_%": smape(y_test.values, pred),
        "error_medio_€": np.mean(y_test.values - pred)
    })

test_results_df = pd.DataFrame(test_results).sort_values("MAE_€")

display(test_results_df)


# 14. Comparación CV vs Test

Un modelo bueno debería tener resultados razonablemente parecidos en validación y test.

Si el test empeora muchísimo respecto a CV, puede indicar:

- cambios en el comportamiento del restaurante,
- pocos datos,
- sobreajuste,
- cambios estacionales,
- variables que no generalizan al futuro.


In [ ]:
comparison = cv_results_df[
    ["modelo", "MAE_CV_mean", "RMSE_CV_mean", "R2_CV_mean"]
].merge(
    test_results_df,
    on="modelo"
)

display(comparison.sort_values("MAE_€"))


# 15. ¿Los modelos están sobreajustando?

Comparamos el MAE de entrenamiento con el MAE de validación.

Si el error de entrenamiento es muchísimo menor que el de validación, hay señales de sobreajuste.


In [ ]:
overfitting = cv_results_df[
    ["modelo", "MAE_train_mean", "MAE_CV_mean"]
].copy()

overfitting["ratio_CV_train"] = (
    overfitting["MAE_CV_mean"] /
    overfitting["MAE_train_mean"]
)

display(overfitting.sort_values("ratio_CV_train"))


# 16. Gráfico: MAE de los cinco modelos

Cuanto menor sea el MAE, mejor.


In [ ]:
plot_df = test_results_df.sort_values("MAE_€")

plt.figure(figsize=(10, 5))
plt.bar(plot_df["modelo"], plot_df["MAE_€"])
plt.ylabel("MAE (€)")
plt.xlabel("Modelo")
plt.title("Comparación de MAE en el test final")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


# 17. Gráfico: RMSE

El RMSE penaliza especialmente los errores grandes.


In [ ]:
plot_df = test_results_df.sort_values("RMSE_€")

plt.figure(figsize=(10, 5))
plt.bar(plot_df["modelo"], plot_df["RMSE_€"])
plt.ylabel("RMSE (€)")
plt.xlabel("Modelo")
plt.title("Comparación de RMSE en el test final")
plt.xticks(rotation=25)
plt.tight_layout()
plt.show()


# 18. Facturación real vs predicción

Mostramos los cinco modelos sobre el mismo período de test.


In [ ]:
plt.figure(figsize=(15, 6))

plt.plot(
    df["fecha"].iloc[split:],
    y_test.values,
    label="Real",
    linewidth=3
)

for name, pred in test_predictions.items():
    plt.plot(
        df["fecha"].iloc[split:],
        pred,
        label=name,
        alpha=0.75
    )

plt.xlabel("Fecha")
plt.ylabel("Facturación (€)")
plt.title("Facturación real vs predicciones — test final")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# 19. Errores diarios del mejor modelo

Analizamos los días en los que el mejor modelo se equivoca más.


In [ ]:
best_model_name_test = test_results_df.iloc[0]["modelo"]
best_pred = test_predictions[best_model_name_test]

resultados = pd.DataFrame({
    "fecha": df["fecha"].iloc[split:].values,
    "facturacion_real": y_test.values,
    "facturacion_predicha": best_pred
})

resultados["error"] = (
    resultados["facturacion_real"] -
    resultados["facturacion_predicha"]
)

resultados["error_abs"] = resultados["error"].abs()

resultados["error_pct"] = np.where(
    resultados["facturacion_real"] != 0,
    resultados["error_abs"] /
    np.abs(resultados["facturacion_real"]) * 100,
    np.nan
)

print("Mejor modelo en test:", best_model_name_test)

display(
    resultados
    .sort_values("error_abs", ascending=False)
    .head(20)
)


# 20. Comprobar si el modelo es razonablemente bueno

No existe un MAE universalmente "bueno": depende del nivel de facturación del restaurante.

Por eso comparamos el MAE con la facturación media del período de test.

Interpretación orientativa:

- error relativo < 10% → muy bueno
- 10–20% → razonable/bueno
- 20–30% → mejorable
- > 30% → predicción débil

Estos umbrales son **orientativos**, no reglas estadísticas.


In [ ]:
mean_revenue_test = y_test.mean()
mae_best = test_results_df.iloc[0]["MAE_€"]

relative_mae = mae_best / mean_revenue_test * 100

print(f"Facturación media diaria del test: {mean_revenue_test:.2f} €")
print(f"MAE del mejor modelo: {mae_best:.2f} €")
print(f"MAE relativo: {relative_mae:.2f}%")

if relative_mae < 10:
    print("Interpretación orientativa: MUY BUENO")
elif relative_mae < 20:
    print("Interpretación orientativa: BUENO / RAZONABLE")
elif relative_mae < 30:
    print("Interpretación orientativa: MEJORABLE")
else:
    print("Interpretación orientativa: DÉBIL")


# 21. Comparación con un baseline ingenuo

Esta comprobación es MUY importante.

Un modelo de Machine Learning no es útil simplemente porque tenga un R² positivo. Debe superar una estrategia sencilla.

Baseline utilizado:

> Predecir que mañana tendremos la misma facturación que el día anterior.

Si el mejor modelo no supera claramente este baseline, hay que revisar las variables y la construcción del problema.


In [ ]:
# Predicción ingenua: facturación del día anterior
baseline_pred = df["facturacion"].shift(1).iloc[split:].values

baseline_mask = ~np.isnan(baseline_pred)

baseline_mae = mean_absolute_error(
    y_test.iloc[np.where(baseline_mask)[0]],
    baseline_pred[baseline_mask]
)

baseline_rmse = np.sqrt(mean_squared_error(
    y_test.iloc[np.where(baseline_mask)[0]],
    baseline_pred[baseline_mask]
))

baseline_r2 = r2_score(
    y_test.iloc[np.where(baseline_mask)[0]],
    baseline_pred[baseline_mask]
)

print(f"Baseline MAE : {baseline_mae:.2f} €")
print(f"Baseline RMSE: {baseline_rmse:.2f} €")
print(f"Baseline R²  : {baseline_r2:.4f}")

print("\nMejor modelo:")
print(f"MAE : {test_results_df.iloc[0]['MAE_€']:.2f} €")
print(f"RMSE: {test_results_df.iloc[0]['RMSE_€']:.2f} €")
print(f"R²  : {test_results_df.iloc[0]['R2']:.4f}")


# 22. Ranking final

El ranking se basa principalmente en MAE, pero también mostramos RMSE, R² y sMAPE.

**Importante:** no elijas el modelo únicamente por R². Para negocio, el MAE en euros es especialmente interpretable.


In [ ]:
ranking = test_results_df.copy()

ranking["ranking_MAE"] = ranking["MAE_€"].rank(method="min")
ranking["ranking_RMSE"] = ranking["RMSE_€"].rank(method="min")
ranking["ranking_R2"] = ranking["R2"].rank(method="min", ascending=False)

ranking["score_ranking"] = (
    ranking["ranking_MAE"] +
    ranking["ranking_RMSE"] +
    ranking["ranking_R2"]
)

ranking = ranking.sort_values(
    ["score_ranking", "MAE_€"]
)

display(ranking)


# 23. Guardar resultados

Guardamos:

- métricas de validación cruzada,
- métricas del test,
- predicciones del mejor modelo,
- errores diarios.


In [ ]:
cv_results_df.to_csv(
    "resultados_validacion_cruzada.csv",
    index=False
)

test_results_df.to_csv(
    "resultados_test_modelos.csv",
    index=False
)

resultados.to_csv(
    "predicciones_mejor_modelo.csv",
    index=False
)

fold_results.to_csv(
    "resultados_fold_a_fold.csv",
    index=False
)

print("Archivos guardados correctamente.")


# 24. Conclusión automática

Esta celda genera una pequeña conclusión para ayudarte a interpretar los resultados.


In [ ]:
best = test_results_df.iloc[0]

print("=" * 70)
print("CONCLUSIÓN")
print("=" * 70)

print(f"Mejor modelo según MAE: {best['modelo']}")
print(f"MAE: {best['MAE_€']:.2f} €")
print(f"RMSE: {best['RMSE_€']:.2f} €")
print(f"R²: {best['R2']:.4f}")
print(f"sMAPE: {best['sMAPE_%']:.2f}%")

if best["MAE_€"] < baseline_mae:
    mejora = (baseline_mae - best["MAE_€"]) / baseline_mae * 100
    print(f"\nEl mejor modelo supera al baseline en MAE en un {mejora:.2f}%.")
else:
    empeora = (best["MAE_€"] - baseline_mae) / baseline_mae * 100
    print(f"\nATENCIÓN: el mejor modelo NO supera al baseline.")
    print(f"Su MAE es un {empeora:.2f}% mayor.")

print("\nPara el TFM, además de estos resultados, conviene analizar")
print("qué variables están disponibles realmente ANTES de cada día.")
